# Verify the visualization pyramid

Acceptance gates for a pyramid built by [`build_pyramid.py`](build_pyramid.py), plus the
final `Cache-Control` pass. Run against the scratch prefix after a shakedown or the
production prefix after a `build_pyramid.yml` run — set `PYRAMID_PREFIX` below. Everything
up to the last section is read-only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import zarr

from build_pyramid import JOBS, dest_store, open_source
from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import store as gsro_store

config = Config('config/global_config_v10.txt')

PYRAMID_PREFIX = 'snowmelt/snowmelt_runoff_onset/global_runoff_onset_v10.0_multiscale_1'
# PYRAMID_PREFIX = 'snowmelt/snowmelt_runoff_onset/scratch_pyramid_shakedown'
SOURCE_TAG = 'v10.0'
N_LEVELS = 10

root = zarr.open_group(dest_store(config, PYRAMID_PREFIX, read_only=True), mode='r')
expected_vars = sorted(v for job in JOBS.values() for v in job
                       if v in root['0'].array_keys())
print(f'variables present: {expected_vars}')

## 1. Structure and attrs lint

Levels exist with iterated floor-halved shapes, the convention attrs are present
(`zarr_conventions`, `multiscales` layout, `proj:`, `spatial:`, `layer_hints`, `provenance`),
and every array carries the right `_FillValue` at both the attr and zarr `fill_value` level —
the v10 store-init bug class, asserted forever.

In [ ]:
attrs = dict(root.attrs)
for key in ['zarr_conventions', 'multiscales', 'proj:code', 'spatial:transform',
            'spatial:shape', 'provenance']:
    assert key in attrs, f'missing root attr: {key}'
assert attrs['proj:code'] == 'EPSG:4326'
prov = attrs['provenance']
print(f"built from {prov['source_store']} @ {prov['source_tag']} "
      f"(snapshot {prov['source_snapshot_id']}), topozarr {prov['topozarr_version']}")

layout = attrs['multiscales']['layout']
assert [entry['asset'] for entry in layout] == [str(i) for i in range(N_LEVELS)]

ny, nx = 204800, 499998
for i in range(N_LEVELS):
    level = root[str(i)]
    for var in expected_vars:
        arr = level[var]
        assert arr.shape[-2:] == (ny, nx), (i, var, arr.shape)
        assert arr.dtype == np.int16
        assert arr.fill_value == -9999, f'{i}/{var} zarr fill_value {arr.fill_value}'
        assert arr.attrs['_FillValue'] == -9999
    ny, nx = ny // 2, nx // 2
print(f'{N_LEVELS} levels x {len(expected_vars)} vars: shapes, dtypes, fills all OK')

## 2. Level 0 vs source — exact

Raw (encoded) equality between pyramid level 0 and the tagged icechunk store on the QC tiles
from `3_quality_check_tiles.ipynb` — level 0 is a copy, so any mismatch is a bug, not a
tolerance question. (Uses one dense water year for the yearly variables.)

In [ ]:
QC_TILES = [(25, 39), (9, 138), (10, 0), (28, 65), (80, 240)]
CHECK_WY = 2020

source_ds, snapshot_id = open_source(config, SOURCE_TAG)
assert snapshot_id == prov['source_snapshot_id'], 'pyramid was built from a different snapshot'
level0 = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                      group='0', zarr_format=3, consolidated=False,
                      mask_and_scale=False, chunks=None)

for row, col in QC_TILES:
    region = gsro_store.tile_region_slices(config, row, col)
    for var in expected_vars:
        src = source_ds[var]
        pyr = level0[var]
        if 'water_year' in src.dims:
            src, pyr = src.sel(water_year=CHECK_WY), pyr.sel(water_year=CHECK_WY)
        src_vals = src.isel(**region).values
        pyr_vals = pyr.isel(**region).values
        assert (src_vals == pyr_vals).all(), f'level-0 mismatch: tile ({row},{col}) {var}'
    n_valid = int((src_vals != -9999).sum())
    print(f'tile ({row},{col}): all vars byte-identical ({n_valid:,} valid px in {var})')

## 3. Cross-level visuals

Decoded views of one region at native / mid / coarse levels — the coarsening should look like
a blur of the same field, never a shift, a striping pattern, or values outside the children's
range. Rainier tile (25,39); swap in any region.

In [ ]:
row, col = 25, 39
var = 'runoff_onset_median' if 'runoff_onset_median' in expected_vars else expected_vars[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, lvl in zip(axes, [0, 2, 4]):
    factor = 2 ** lvl
    ds_l = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                        group=str(lvl), zarr_format=3, consolidated=False, chunks=None)
    da = ds_l[var]
    if 'water_year' in da.dims:
        da = da.sel(water_year=CHECK_WY)
    region = gsro_store.tile_region_slices(config, row, col)
    sub = da.isel(latitude=slice(region['latitude'].start // factor,
                                 region['latitude'].stop // factor),
                  longitude=slice(region['longitude'].start // factor,
                                  region['longitude'].stop // factor))
    sub.plot.imshow(ax=ax, vmin=110, vmax=270, cmap='viridis', add_colorbar=(lvl == 4))
    ax.set_title(f'level {lvl} ({80 * factor:.0f} m)')
    ax.set_aspect('equal')
fig.suptitle(f'{var}, tile ({row},{col})')
fig.tight_layout()

## 4. Global sanity render

The static-map preview: one coarse level, whole world, decoded. This is essentially what the
`../global/` figure notebooks will consume instead of the old coarsened store.

In [ ]:
ds_l5 = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                     group='5', zarr_format=3, consolidated=False, chunks=None)
da = ds_l5[var]
if 'water_year' in da.dims:
    da = da.sel(water_year=CHECK_WY)
da = da.load()

fig, ax = plt.subplots(figsize=(16, 6))
da.plot.imshow(ax=ax, vmin=110, vmax=270, cmap='viridis')
ax.set_title(f'{var} @ level 5 (~2.6 km)')
print(f'valid px at level 5: {int(da.notnull().sum()):,}')

## 5. Cache headers

`Cache-Control: public, max-age=31536000, immutable` on every blob under the prefix — safe
because cache-busting is by prefix version, not by mutation. Re-run after any job rerun.
**Writes blob properties**; everything above this cell is read-only.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from azure.storage.blob import ContainerClient, ContentSettings

container, prefix = PYRAMID_PREFIX.split('/', 1)
account_url = f'https://{config.azure_storage_account}.blob.core.windows.net'
client = ContainerClient(account_url, container, credential=config.sas_token)

CACHE = 'public, max-age=31536000, immutable'
blobs = [b for b in client.list_blobs(name_starts_with=prefix + '/')]
todo = [b for b in blobs if (b.content_settings.cache_control or '') != CACHE]
print(f'{len(blobs):,} blobs, {len(todo):,} need the header')


def _set_header(blob):
    settings = ContentSettings(cache_control=CACHE,
                               content_type=blob.content_settings.content_type)
    client.get_blob_client(blob.name).set_http_headers(content_settings=settings)


with ThreadPoolExecutor(max_workers=16) as pool:
    list(pool.map(_set_header, todo))
print('done')